# DEH 30-Day PySpark Challenge
## Day 10 — Handling Nulls

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — Create a DataFrame with nulls

In [ ]:
data = [
    ("O0001", "C001", 1299.99, 2,    "Delivered", "East"),
    ("O0002", "C002",  449.99, 1,    "Delivered", "West"),
    ("O0003", None,    349.99, 4,    "Shipped",   "Midwest"),
    ("O0004", "C004",    None, 3,    "Delivered", None),
    ("O0005", "C005",   89.99, None, None,        "South"),
    ("O0006", None,      None, 2,    "Cancelled", "East"),
    ("O0007", "C007",  199.99, 1,    "Delivered", "West")
]

schema = StructType([
    StructField("order_id",    StringType(),  True),
    StructField("customer_id", StringType(),  True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("status",      StringType(),  True),
    StructField("region",      StringType(),  True)
])

df = spark.createDataFrame(data, schema)
df.show()

## Step 5 — isNull() and isNotNull()

In [ ]:
# Filter rows where customer_id is null
print("Rows where customer_id is null:")
df.filter(F.col("customer_id").isNull()).show()

# Filter rows where customer_id is NOT null
print("Rows where customer_id is NOT null:")
df.filter(F.col("customer_id").isNotNull()).show()

In [ ]:
# Count nulls per column — very useful for data quality checks
null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
print("Null counts per column:")
null_counts.show()

## Step 6 — dropna()

In [ ]:
# Drop rows where ANY column is null
print(f"Original rows : {df.count()}")
print(f"After dropna() any: {df.dropna().count()}")
df.dropna().show()

# Drop rows where specific columns are null
print(f"After dropna on customer_id + unit_price: {df.dropna(subset=['customer_id', 'unit_price']).count()}")
df.dropna(subset=["customer_id", "unit_price"]).show()

## Step 7 — fillna()

In [ ]:
# Fill with a dictionary — different value per column
df_filled = df.fillna({
    "customer_id": "Unknown",
    "unit_price":  0.0,
    "quantity":    1,
    "status":      "Pending",
    "region":      "Unknown"
})

df_filled.show()

# Verify no nulls remain
null_counts_after = df_filled.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_filled.columns
])
print("Null counts after fillna:")
null_counts_after.show()

## Step 8 — F.coalesce()

In [ ]:
# coalesce — pick first non-null from multiple columns
data = [
    ("O0001", "primary@email.com", None),
    ("O0002", None,                "backup@email.com"),
    ("O0003", "main@email.com",    "alt@email.com"),
    ("O0004", None,                None)
]

email_df = spark.createDataFrame(
    data,
    ["order_id", "primary_email", "backup_email"]
)

email_df.withColumn(
    "contact_email",
    F.coalesce(
        F.col("primary_email"),
        F.col("backup_email"),
        F.lit("no-email@unknown.com")
    )
).show()

---
## Assignment — Day 10

---

### Task 1
Create a DataFrame with at least 5 rows and intentional nulls across multiple columns.  
Count the number of nulls in each column using the `when(isNull()).sum()` pattern.

In [ ]:
# Task 1 — Write your code here


### Task 2
Using your DataFrame from Task 1, drop rows where any column is null.  
Then drop rows where only specific columns are null.  
Compare the row counts.

In [ ]:
# Task 2 — Write your code here


### Task 3
Using `fillna()` with a dictionary, fill nulls:  
- String columns → "Unknown"  
- Numeric columns → 0  

Show the result — no nulls should remain.

In [ ]:
# Task 3 — Write your code here


### Task 4
Create a DataFrame with `billing_address` and `shipping_address` columns where some rows have one or both as null.  
Use `F.coalesce()` to create a `delivery_address` column that picks the first non-null address, falling back to `"Address Unknown"`.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*